## Preamble

In [58]:
import sys
from pathlib import Path

sys.path.insert(0,
                str(Path.cwd().parent))

In [59]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

from src.encode import *
from src.paths import *
from src.tiling import get_tiles

In [60]:
TISSUE_EMBED_PATH = os.path.join(DATA_DIR,"tissue_embeddings/")
EXHAUSTIVE_EMBED_PATH = os.path.join(DATA_DIR,"exhaustive_embeddings/")

In [61]:
SEED = 0

## Code

In [62]:
df = pd.read_parquet(find_input("slide_inventory.parquet"))
df.head()

slide_inventory.parquet: /home/aztrek/projects/Prostate-Cancer-Grading/data/derived/slide_inventory.parquet


,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty
0,0005f7aaab2800f6170c399693a96917,0+0,0,karolinska,True,True,None,None,3,"[27648, 29440]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",96.895192
1,000920ad0b612851f8e01bcc880d9b3d,0+0,0,karolinska,True,True,None,None,3,"[15360, 13312]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",94.424705
2,0018ae58b01bdadc8e347995b69f99aa,4+4,4,radboud,True,True,None,None,3,"[5888, 25344]","[1.0, 4.0, 16.0]","[0.4861876369654638, 0.4861876369654638]",82.860363
3,001c62abd11fa4b57bf7a6c603a11bb9,4+4,4,karolinska,True,True,None,None,3,"[23904, 28664]","[1.0, 4.0, 16.00223338916806]","[0.5031982437947761, 0.5031982437947761]",95.209238
4,001d865e65ef5d2579c190a0e0350d8f,0+0,0,karolinska,True,True,None,None,3,"[28672, 34560]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",94.389209


In [63]:
df["unified_label"] = list(zip(df["isup_grade"],df["provider"]))
df = df[["slide_id","unified_label"]]
df.head()

,slide_id,unified_label
0,0005f7aaab2800f6170c399693a96917,"(0, karolinska)"
1,000920ad0b612851f8e01bcc880d9b3d,"(0, karolinska)"
2,0018ae58b01bdadc8e347995b69f99aa,"(4, radboud)"
3,001c62abd11fa4b57bf7a6c603a11bb9,"(4, karolinska)"
4,001d865e65ef5d2579c190a0e0350d8f,"(0, karolinska)"


In [64]:
selected_stratified_portion , _ = train_test_split(
    df,
    train_size = 2000,
    stratify = df["unified_label"],
    random_state = 42
)
selected_stratified_portion

,slide_id,unified_label
3191,4ff3969ded27b5fa1ea6dbeb7bdd8196,"(0, karolinska)"
8768,d3abadcb28a19cd4c68edf9be745c4b6,"(0, karolinska)"
5787,8dfe44375f062a6de5aac69741ae7dbe,"(0, karolinska)"
5083,7d86f8993e77d4f99ed19b2965d5067d,"(2, radboud)"
8557,ce8d8607d29aff038a6e27528a9789ee,"(3, karolinska)"
...,...,...
6506,9ecba0a5f5caa828df43867455405dd7,"(3, radboud)"
3466,56a882d88e118dcb77a63bfdbdf0b716,"(3, radboud)"
2331,3a450c205cf27f0348b46d8329c13fc6,"(0, karolinska)"
10273,f83625741b5c20294cf356bbb4060927,"(1, karolinska)"


In [65]:
encoder = load_encoder_offline(weights_path())
encoder = prepare(encoder)


In [66]:
for id in selected_stratified_portion["slide_id"]:
    torch.save(encode_tiles(encoder, get_tiles(slide_path(id))[0]), os.path.join(TISSUE_EMBED_PATH,f"{id}.pt"))

AttributeError: 'list' object has no attribute 'size'

## Exhaustive grid: the floor to beat

The second pilot set uses a plain evenly spaced grid over the whole slide, and ignores tissue entirely. Most of its tiles land on blank glass, which is the point: it is a deliberately poor way to choose tiles. If choosing tiles by tissue does not clearly beat it, the tissue step is not doing its job.

Only the tile *positions* differ from the tissue set. The read level, the tile size, the reading (`read_tiles`), the encoder and the transform (`encode_tiles`) are the same, so any difference in score comes from where the tiles were placed.

In [ ]:
import math

import numpy as np
import openslide

from src.tiling import READ_LEVEL, TILE_PER_SLIDE_CAP, TILE_SIZE, read_tiles


def get_grid_tiles(slide_path, n=TILE_PER_SLIDE_CAP):
    """Up to n tiles on an evenly spaced grid over the whole slide, tissue or not.

    1. Cut the read level into whole TILE_SIZE cells: n_cols across, n_rows down.
    2. Split n between columns and rows in proportion to the slide's shape, so
       the spacing is about the same both ways. A long, narrow slide gets few
       columns and many rows. Neither can exceed the cells that exist.
    3. Spread the chosen columns and rows evenly from edge to edge (linspace),
       and take every (column, row) pair, in reading order.
    4. Convert cells to level-0 (x, y): a cell is TILE_SIZE pixels at READ_LEVEL,
       which is TILE_SIZE x that level's downsample in level-0 pixels.
    5. Read the tiles with the same read_tiles as the tissue set, and close the
       slide in a finally block.

    Returns tiles and coords, like get_tiles. There is no randomness, so a slide
    always gives the same tiles.
    """
    slide = openslide.OpenSlide(slide_path)
    try:
        width, height = slide.level_dimensions[READ_LEVEL]
        n_cols, n_rows = width // TILE_SIZE, height // TILE_SIZE

        k_cols = min(max(1, round(math.sqrt(n * n_cols / n_rows))), n_cols)
        k_rows = min(n // k_cols, n_rows)

        cols = np.unique(np.linspace(0, n_cols - 1, k_cols).round().astype(int))
        rows = np.unique(np.linspace(0, n_rows - 1, k_rows).round().astype(int))
        cells = np.array([(col, row) for row in rows for col in cols])

        coords = np.round(cells * TILE_SIZE * slide.level_downsamples[READ_LEVEL]).astype(np.int32)
        tiles = read_tiles(slide, coords, READ_LEVEL)
    finally:
        slide.close()
    return tiles, coords

In [ ]:
os.makedirs(EXHAUSTIVE_EMBED_PATH, exist_ok=True)

for id in selected_stratified_portion["slide_id"]:
    torch.save(encode_tiles(encoder, get_grid_tiles(slide_path(id))[0]), os.path.join(EXHAUSTIVE_EMBED_PATH,f"{id}.pt"))